# Comparative epitope mapping

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoLlama/epitope_align/blob/claude/antibody-epitope-identification-ervnyf/notebooks/epitope_mapping.ipynb)

Work out **where an antibody probably binds** from the fact that it binds some
species' versions of a protein and not others.

You do not need to know how to code. There are four steps, and you run each one
by pressing the **▶ play button** on the left of the grey box below it.

1. **Set up** - installs the tool (about two minutes, once per session)
2. **Check it works** - runs a built-in example with a known answer
3. **Enter your data** - fill in the boxes
4. **Run and read the results**

Nothing here touches your own computer, and nothing you type is sent anywhere
except to UniProt and the structure databases, to look up the entries you name.

---

## Step 1 - Set up

Press play. It installs the tool straight from GitHub, along with two optional
helpers (MAFFT for better multi-species alignments, DSSP for secondary
structure). Wait for the word **Ready** before moving on.


In [ ]:
# @title Install (press play, then wait for 'Ready') { display-mode: "form" }

REPO = 'https://github.com/NanoLlama/epitope_align.git'
BRANCH = 'claude/antibody-epitope-identification-ervnyf'  # change to 'main' once this branch is merged

import subprocess

print('Installing MAFFT and DSSP ...')
subprocess.run('apt-get -qq update', shell=True, capture_output=True)
subprocess.run('apt-get -qq install -y mafft dssp', shell=True, capture_output=True)

print('Installing epitope-map ...')
install = subprocess.run(
    f'pip install -q "git+{REPO}@{BRANCH}"',
    shell=True, capture_output=True, text=True,
)
if install.returncode != 0:
    print(install.stdout[-2000:])
    print(install.stderr[-2000:])
    raise SystemExit(
        'Install failed. Check the branch name above still exists on GitHub.'
    )

import epitope_map

def _have(tool):
    return 'yes' if subprocess.run(f'which {tool}', shell=True,
                                   capture_output=True).returncode == 0 else 'no'

print(f'\nReady: epitope-map {epitope_map.__version__}')
print('MAFFT:', _have('mafft'))
print('DSSP: ', _have('mkdssp'))


---

## Step 2 - Check it works

This runs a made-up example where the answer is known in advance: six invented
species, three that bind and three that do not, and an epitope deliberately
planted at residues **65, 66, 68, 70, 72, 114, 116 and 118**.

If the top patch printed below is exactly those eight residues, everything is
working.


In [ ]:
!epitope-map --demo --outdir /content/demo-run


---

## Step 3 - Enter your data

Fill in the boxes on the right, then press play. Nothing is analysed yet - this
cell only checks that what you typed makes sense, and prints it back so you can
confirm it was read correctly.

Note that each box is a **single line**: you cannot press Enter inside one, so
lists are separated with commas or semicolons.

**Sequences.** Easiest is UniProt accession numbers, one per species, written as
`label=ACCESSION` and separated by commas:

```
mouse=Q61503,rat=P21590,human=P21589
```

Look each one up at <https://www.uniprot.org> by searching your protein plus the
species name; the accession is the code like `P21589` near the top of the entry.
The labels are yours to choose - they only have to match the binding box below.

*Alternatively* tick `use_uploaded_fasta` and you will be asked for a FASTA file.

**Binding.** The same labels, each with its result, on one line:

```
mouse=binder, rat=binder, human=non_binder, marmoset=non_binder
```

Semicolons work too, and so does `mouse,binder; rat,binder` if you prefer commas
between the species and its result. `yes`/`no` are accepted in place of
`binder`/`non_binder`. Use `unknown` for a species you have a sequence for but
no binding data - it will be shown but not used for scoring.

You need at least one binder and one non-binder. Three scored species is the
minimum; six to eight is far better, because each extra informative species
roughly halves the shortlist.

**Reference.** The species the structure belongs to. It must be one that binds -
all numbering in the results refers to it.

**Structure.** Any of:
- an AlphaFold accession like `AF-P21589-F1` (use the reference species' UniProt
  accession; every UniProt page links its AlphaFold model)
- a Protein Data Bank ID like `4H2I` if an experimental structure exists
- tick `use_uploaded_structure` to upload your own `.pdb`/`.cif` file

**Ectodomain** (optional). If you only care about part of the protein - say the
part outside the cell - enter it as `25-240`, in the structure's own numbering.
Leave blank to analyse everything.


In [ ]:
# @title Your inputs { display-mode: "form" }
# @markdown ### Sequences &mdash; `label=ACCESSION`, comma separated
sequences = "mouse=REPLACE_ME,rat=REPLACE_ME,human=REPLACE_ME"  # @param {type:"string"}
use_uploaded_fasta = False  # @param {type:"boolean"}

# @markdown ### Binding results &mdash; `label=binder` / `label=non_binder`, comma separated
binding = "mouse=binder, rat=binder, human=non_binder"  # @param {type:"string"}

# @markdown ### Reference species (must be one that binds)
reference = "mouse"  # @param {type:"string"}

# @markdown ### Structure of the reference species
structure = "AF-REPLACE_ME-F1"  # @param {type:"string"}
use_uploaded_structure = False  # @param {type:"boolean"}

# @markdown ### Optional
ectodomain = ""  # @param {type:"string"}
chain = ""  # @param {type:"string"}

import pathlib

from epitope_map.io_seq import InputError
from epitope_map.notebook import (
    check_panel, parse_binding_calls, sequence_labels, summarise, write_binding_csv,
)

work = pathlib.Path('/content/my-run')
work.mkdir(parents=True, exist_ok=True)

if use_uploaded_fasta:
    from google.colab import files
    print('Choose your FASTA file:')
    uploaded = files.upload()
    sequences_arg = str(work / list(uploaded)[0])
    pathlib.Path(sequences_arg).write_bytes(list(uploaded.values())[0])
else:
    sequences_arg = sequences

if use_uploaded_structure:
    from google.colab import files
    print('Choose your structure file (.pdb or .cif):')
    uploaded = files.upload()
    structure_arg = str(work / list(uploaded)[0])
    pathlib.Path(structure_arg).write_bytes(list(uploaded.values())[0])
else:
    structure_arg = structure

try:
    calls = parse_binding_calls(binding)
except InputError as problem:
    raise SystemExit(f'Could not read the binding box: {problem}')

binding_path = write_binding_csv(calls, work / 'binding.csv')
problems, notes = check_panel(calls, sequence_labels(sequences_arg), reference)

print(summarise(calls, reference))
print(f'\nstructure : {structure_arg}')
print(f'sequences : {sequences_arg}')
print(f'range     : {ectodomain.strip() or "whole chain"}')

for note in notes:
    print(f'\nnote: {note}')
if problems:
    print('\n' + '=' * 60)
    for problem in problems:
        print(f'FIX THIS: {problem}')
    print('=' * 60)
    raise SystemExit('Correct the boxes above and press play again.')
print('\nInputs look consistent. Move on to Step 4.')


---

## Step 4 - Run it

This fetches anything it needs, aligns the sequences, measures the surface of the
structure and ranks the candidate patches. A small protein takes under a minute.

Read the **warnings** it prints. They are not errors - they are the honest
limitations of your particular run, and the biggest one is usually that your
binders and non-binders are two separate branches of the family tree, which
leaves a lot of irrelevant differences looking meaningful.


In [ ]:
import shlex, subprocess

command = [
    'epitope-map',
    '--sequences', sequences_arg,
    '--binding', str(binding_path),
    '--reference', reference,
    '--structure', structure_arg,
    '--outdir', '/content/my-run/results',
]
if ectodomain.strip():
    command += ['--ectodomain', ectodomain.strip()]
if chain.strip():
    command += ['--chain', chain.strip()]

print(' '.join(shlex.quote(part) for part in command), '\n')
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('--- it stopped with this message ---')
    print(result.stderr)


---

## Step 5 - Read the results

The report below is the thing to read. The key sections are *How much signal is
there?* (whether to trust any of it), *Top candidate patches* (the answer), and
*Suggested next experiments* (what to make in the lab).

Remember what this is: a **ranked shortlist of hypotheses to test**, not a
prediction. The top patch being right is something your chimeras and mutants
decide, not the software.


In [ ]:
from IPython.display import Markdown, display
import pandas as pd, pathlib

results = pathlib.Path('/content/my-run/results')
display(Markdown((results / 'report.md').read_text()))


### The candidate patches, as a table


In [ ]:
patches = pd.read_csv(results / 'patches.tsv', sep='\t')
display(patches[['patch_id', 'rank_raw', 'rank_normalized', 'n_residues',
                 'residues', 'total_score', 'mean_rsa', 'spread_A', 'flags']])


### The mutants worth making

`gain_of_binding` rows are the convincing experiment: they put the binder's
residue into a species that does *not* bind, so a positive result cannot be
explained away as a badly folded protein.


In [ ]:
mutants = pd.read_csv(results / 'mutants.tsv', sep='\t')
display(mutants[['patch_id', 'direction', 'background_species', 'mutation',
                 'numbering', 'grantham', 'rsa', 'priority']].head(20))


### Download everything

Includes `session.pml`, which opens the structure in PyMOL with the candidate
patches coloured in.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/epitope-results', 'zip', results)
files.download('/content/epitope-results.zip')
